# W3 single-phase 3D CNN

**Scientific purpose.** Define the historical portal-venous comparator: a randomly
initialized MONAI 3D ResNet-18 receiving one independently centered C2 crop with shape
`1 x 96 x 96 x 96`.

**Runnability classification:** historical training workflow requiring private processed
arrays and private patient-level splits. Definitions are inspectable, but the distributed
repository does not launch training automatically.

**Inputs:** private C2 crop arrays and patient-level folds.
**Private outputs when deliberately run:** fold checkpoints and patient-level OOF/test
probabilities. These must never be written into the public repository. W3 fold 1 from the
historical run was not retained, so exact complete historical inference cannot be rebuilt.


In [ ]:
from pathlib import Path
import sys


def locate_repository(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = locate_repository()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from liver_tumor_pipeline.config import load_config, require_path

CONFIG_PATH = REPO_ROOT / "configs" / "paths.yaml"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(
        "Create an untracked configs/paths.yaml from configs/paths.example.yaml and "
        "set the documented environment variables before running this workflow."
    )

CONFIG = load_config(CONFIG_PATH)
PRIVATE_ARTIFACT_ROOT = require_path(CONFIG, "private_artifact_root", must_exist=False)
OUTPUT_ROOT = require_path(CONFIG, "output_root", must_exist=False)

import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from monai.networks.nets import resnet18
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

SPLIT_ROOT = PRIVATE_ARTIFACT_ROOT / "splits"
PROCESSED_ROOT = PRIVATE_ARTIFACT_ROOT / "processed" / "internal"
for required in (SPLIT_ROOT / "train_val_test.json", SPLIT_ROOT / "fold_0.json"):
    if not required.is_file():
        raise FileNotFoundError(
            "Private split definitions are unavailable; run notebook 01 in authorized storage."
        )
if not PROCESSED_ROOT.is_dir():
    raise FileNotFoundError(
        "Private processed arrays are unavailable; run notebook 02 in authorized storage."
    )

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True


## Dataset and augmentation

C2 is channel index 2 in the locked order P, C1, C2, C3. Training augmentation consists
of two random in-plane flips and intensity scale/shift jitter. There was no random crop
shift and no validation or test augmentation.


In [ ]:
class SinglePhaseDataset(Dataset):
    def __init__(self, patient_keys, processed_root: Path, *, train: bool = False):
        self.patient_keys = list(patient_keys)
        self.processed_root = Path(processed_root)
        self.train = train

    def __len__(self):
        return len(self.patient_keys)

    @staticmethod
    def normalize_nonzero(volume: np.ndarray) -> np.ndarray:
        output = np.zeros_like(volume, dtype=np.float32)
        mask = volume > 0
        if mask.sum() < 100:
            return volume.astype(np.float32)
        values = volume[mask]
        output[mask] = (values - values.mean()) / (values.std() + 1e-6)
        return output

    @staticmethod
    def augment(volume: np.ndarray) -> np.ndarray:
        if random.random() < 0.5:
            volume = volume[::-1].copy()
        if random.random() < 0.5:
            volume = volume[:, ::-1].copy()
        if random.random() < 0.5:
            scale = 1.0 + (random.random() - 0.5) * 0.2
            shift = (random.random() - 0.5) * 0.2
            nonzero = volume != 0
            volume[nonzero] = volume[nonzero] * scale + shift
        return volume

    def __getitem__(self, index):
        patient_key = self.patient_keys[index]
        path = self.processed_root / f"{patient_key}.npz"
        if not path.is_file():
            raise FileNotFoundError("A required private processed patient array is unavailable.")
        with np.load(path, allow_pickle=False) as archive:
            volume = archive["ct"][2].astype(np.float32)
            label = int(archive["label"])
        volume = self.normalize_nonzero(volume)
        if self.train:
            volume = self.augment(volume)
        return torch.from_numpy(volume[None].astype(np.float32)), label, patient_key


def make_w3_model() -> nn.Module:
    return resnet18(
        spatial_dims=3,
        n_input_channels=1,
        num_classes=3,
        feed_forward=True,
    )


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

BATCH_SIZE = 4
NUM_EPOCHS = 40
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5


def evaluate(model, loader, device):
    model.eval()
    labels, probabilities = [], []
    with torch.no_grad():
        for images, targets, _patient_keys in loader:
            logits = model(images.to(device))
            probabilities.append(torch.softmax(logits, dim=1).cpu().numpy())
            labels.append(targets.numpy())
    y_true = np.concatenate(labels)
    probs = np.concatenate(probabilities)
    predictions = probs.argmax(axis=1)
    return {
        "auc": roc_auc_score(y_true, probs, multi_class="ovr", average="macro"),
        "accuracy": accuracy_score(y_true, predictions),
        "macro_f1": f1_score(y_true, predictions, average="macro"),
        "labels": y_true,
        "probabilities": probs,
        "predictions": predictions,
    }


def train_one_fold(model, train_loader, validation_loader, development_labels, device):
    # Historical class weights were computed from all 222 development labels, not only the
    # fold-training subset. This is preserved and disclosed as a CV-isolation limitation.
    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1, 2]),
        y=np.asarray(development_labels),
    )
    criterion = nn.CrossEntropyLoss(
        weight=torch.as_tensor(weights, dtype=torch.float32, device=device)
    )
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    best_auc = -np.inf
    best_state = None
    best_metrics = None
    for _epoch in range(NUM_EPOCHS):
        model.train()
        for images, targets, _patient_keys in train_loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(images.to(device)), targets.to(device))
            loss.backward()
            optimizer.step()
        metrics = evaluate(model, validation_loader, device)
        if metrics["auc"] > best_auc:
            best_auc = metrics["auc"]
            best_metrics = metrics
            best_state = {
                key: value.detach().cpu().clone() for key, value in model.state_dict().items()
            }
    if best_state is None or best_metrics is None:
        raise RuntimeError("No finite validation checkpoint was selected.")
    return best_state, best_metrics


## Deliberate execution boundary

Call `train_one_fold` only in an authorized training environment after constructing
fold-specific loaders from the private split files. Checkpoint selection uses the outer
validation fold that also supplies OOF predictions, which can make CV/stacking estimates
optimistic. It is not patient leakage and does not involve the held-out 56-patient set.
The verified aggregate W3 values are released in
`results/aggregate/internal_performance.csv`.
